# HypotheSAEs on Drugs.com reviews: what distinguishes positive from negative medication reviews?

Runs the HypotheSAEs pipeline (embed → SAE → select neurons → interpret → evaluate) on the **UCI Drug Review dataset** (Gräßer et al., 2018; CC BY 4.0): patient reviews of medications from Drugs.com, each with a 1–10 rating. Label = 1 for positive reviews (rating 9–10), 0 for negative reviews (rating 1–4); middle ratings are dropped.

Same setup as `quickstart_hippocorpus.ipynb`:
- **Embeddings + SAE** in this Colab session (GPU runtime: *Runtime → Change runtime type → T4 GPU*).
- **LLM** (interpretation + annotation): Qwen3-8B via vLLM on Modal (`modal/vllm_server.py`), with `prompt_suffix=" /no_think"` for annotation.

Needs the Colab secret `MODAL_VLLM_KEY` (key icon in the left sidebar), with notebook access enabled.

In [ ]:
!pip install -q git+https://github.com/psnandini/HypotheSAEs.git
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import userdata, files

os.environ["OPENAI_BASE_URL"] = "https://psnandini--hypothesaes-vllm-serve.us-east.modal.run/v1"
os.environ["OPENAI_KEY_SAE"] = userdata.get("MODAL_VLLM_KEY")
os.environ["EMB_CACHE_DIR"] = "/content/emb_cache"

import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.llm_api import get_completion

MODEL = "Qwen/Qwen3-8B"
INTERPRETER_MODEL = ANNOTATOR_MODEL = MODEL
OUT = "/content/results"   # every result table is saved here and downloaded right away
os.makedirs(OUT, exist_ok=True)

**Wake up the LLM server.** It scales to zero when idle, so the first call can take 2–3 minutes.

In [ ]:
import time
t = time.time()
print(get_completion(prompt="Reply with the single word: ready /no_think", model=MODEL))
print(f"{time.time() - t:.0f} s")

## 0. Load the drug reviews

The same review often appears several times (listed under a brand and a generic name), so we drop duplicate texts before sampling; otherwise a review could land in both training and held-out data. We then take a balanced random sample of 3,000 positive and 3,000 negative reviews and split it 70/10/20 into train / validation / held-out.

In [ ]:
!wget -q -nc https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
!unzip -q -o drugsCom_raw.zip -d drugs

df = pd.concat([pd.read_csv(f"drugs/{f}", sep="\t") for f in ["drugsComTrain_raw.tsv", "drugsComTest_raw.tsv"]])
df["review"] = df.review.map(html.unescape).str.strip().str.strip('"').str.strip()
df = df.drop_duplicates(subset="review")
print(f"{len(df)} unique reviews")

df = df[(df.rating >= 9) | (df.rating <= 4)].copy()
df["label"] = (df.rating >= 9).astype(int)
df = df.groupby("label").sample(n=3000, random_state=0).sample(frac=1, random_state=0).reset_index(drop=True)
df["n_words"] = df.review.str.split().str.len()

n = len(df)
df["split"] = np.select([df.index < 0.7 * n, df.index < 0.8 * n], ["train", "val"], "holdout")
print(df.groupby(["split", "label"]).size().unstack())

train_df, val_df, holdout_df = (df[df.split == s].reset_index(drop=True) for s in ["train", "val", "holdout"])
texts, labels = train_df.review.tolist(), train_df.label.values
val_texts = val_df.review.tolist()

Quick checks: review length by class, and the most common conditions in the sample.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for lab, g in df.groupby("label"):
    name = "positive (9-10)" if lab else "negative (1-4)"
    ax.hist(g.n_words.clip(upper=300), bins=50, alpha=0.5, label=f"{name}, median {g.n_words.median():.0f}")
ax.set_xlabel("words per review"); ax.set_ylabel("reviews"); ax.legend(); plt.tight_layout(); plt.show()
print(df.condition.value_counts().head(10))

## 1. Text embeddings

`nomic-ai/modernbert-embed-base`, cached in `EMB_CACHE_DIR`.

In [ ]:
EMBEDDER = "nomic-ai/modernbert-embed-base"
text2embedding = get_local_embeddings(texts + val_texts, model=EMBEDDER, batch_size=64, cache_name="drugs_modernbert")
train_embeddings = np.stack([text2embedding[t] for t in texts])
val_embeddings = np.stack([text2embedding[t] for t in val_texts])
print(train_embeddings.shape, val_embeddings.shape)

## 2. Train the SAE

Same settings as the Hippocorpus run (M=256, K=8, Matryoshka prefixes [32, 256]); ~4.2K training reviews.

In [ ]:
sae = train_sae(
    embeddings=train_embeddings, M=256, K=8, matryoshka_prefix_lengths=[32, 256],
    checkpoint_dir="/content/checkpoints/drugs", val_embeddings=val_embeddings,
)

## 3. Sanity check: interpret a few random neurons

The instructions ask for specific content rather than overall satisfaction, since "the reviewer likes the drug" would separate the classes but explain nothing.

In [ ]:
TASK_SPECIFIC_INSTRUCTIONS = """All of the texts are patient reviews of a medication, posted on Drugs.com.
Features should describe a specific property of the review's content, such as a particular symptom, side effect, or experience,
rather than general satisfaction with the drug. For example:
- "mentions a specific dosage in milligrams"
- "mentions taking the medication for several years"
"""

INTERPRET_LLM_KWARGS = {}                                # thinking on for interpretation
ANNOTATION_LLM_KWARGS = {"prompt_suffix": " /no_think"}  # thinking off for yes/no annotation

random_results = interpret_sae(
    texts=texts, embeddings=train_embeddings, sae=sae,
    interpreter_model=INTERPRETER_MODEL, n_random_neurons=5, print_examples_n=3,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS, interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

## 4. Generate hypotheses

Select the 20 neurons most correlated with the label, interpret each, and score fidelity. The table is saved and downloaded as soon as it is ready.

In [ ]:
results = generate_hypotheses(
    texts=texts, labels=labels, embeddings=train_embeddings, sae=sae,
    interpreter_model=INTERPRETER_MODEL, annotator_model=ANNOTATOR_MODEL,
    selection_method="correlation", n_selected_neurons=20, n_candidate_interpretations=1,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS, annotation_llm_kwargs=ANNOTATION_LLM_KWARGS,
    n_workers_interpretation=8, n_workers_annotation=32,
)
pd.set_option("display.max_colwidth", None)
display(results.sort_values("target_correlation", ascending=False))
results.to_csv(f"{OUT}/drugs_hypotheses.csv", index=False)
files.download(f"{OUT}/drugs_hypotheses.csv")

## 5. Evaluate on held-out reviews

Annotate each hypothesis on the 1,200 held-out reviews and test whether it predicts positive vs negative.

In [ ]:
metrics, evaluation_df = evaluate_hypotheses(
    hypotheses_df=results, texts=holdout_df.review.tolist(), labels=holdout_df.label.values,
    annotator_model=ANNOTATOR_MODEL, annotation_llm_kwargs=ANNOTATION_LLM_KWARGS, n_workers_annotation=32,
)
evaluation_df.to_csv(f"{OUT}/drugs_holdout_eval.csv", index=False)
files.download(f"{OUT}/drugs_holdout_eval.csv")
display(evaluation_df)
print(f"AUROC: {metrics['auroc']:.3f}")
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} (p < {metrics['Significant'][2]})")